[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YangKCLab/social-media-analysis/blob/main/docs/topics/data-collection/4chan_collect_board.ipynb)

# 4chan API: collect a whole board

Collect every thread on a board, with all of its replies, by polling the archive. The notebook runs one round. In a real collector the same round runs from `cron` every few minutes for as long as the study lasts.

The 4chan API is read-only JSON. It needs no account and no key. The only
dependency is `requests`, which Google Colab has preinstalled.

Rules from the [API documentation](https://github.com/4chan/4chan-API):
at most one request per second, poll a thread no more often than every 10
seconds, and disclose 4chan as the source of anything you publish from it.
Boards can contain offensive and not-safe-for-work content. Read the
"Research considerations" section on the topic page before collecting.

## Why the archive

Two ways to get every thread on a board:

1. **Poll the catalog.** Easy, but `last_replies` holds only five replies, so
   you never get complete threads, and a thread that appears and expires
   between two polls is missed.
2. **Poll the archive.** An archived thread is complete and no longer changes.
   Fetch each newly archived thread once, before it is deleted, and you have
   the whole board. The cost is a delay: you see a thread only after it expires.

Approach 2 needs three things: a record of the thread IDs already fetched, a
polling interval shorter than the time between archiving and deletion, and a
scheduler that keeps running when you are not looking.

In [1]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import requests

In [2]:
BOARD = "g"
OUT_FILE = Path(f"4chan_{BOARD}_threads.jsonl")   # one thread per line
SEEN_FILE = Path(f"4chan_{BOARD}_seen.json")      # thread IDs already fetched
MAX_PER_ROUND = 3                                  # small for the demo; drop the cap in production
HEADERS = {"User-Agent": "cs415-course-demo"}

In [3]:
def load_seen():
    if SEEN_FILE.exists():
        return set(json.loads(SEEN_FILE.read_text()))
    return set()

def save_seen(seen):
    SEEN_FILE.write_text(json.dumps(sorted(seen)))

seen = load_seen()
len(seen)

0

In [4]:
def get_archive(board):
    resp = requests.get(f"https://a.4cdn.org/{board}/archive.json", headers=HEADERS)
    resp.raise_for_status()
    return resp.json()

def get_thread(board, op_id):
    resp = requests.get(f"https://a.4cdn.org/{board}/thread/{op_id}.json", headers=HEADERS)
    if resp.status_code == 404:
        return None          # deleted between the archive poll and now
    resp.raise_for_status()
    return resp.json()["posts"]

In [5]:
# One round: find newly archived threads and fetch them
archived = get_archive(BOARD)
new_ids = [op_id for op_id in archived if op_id not in seen]
len(archived), len(new_ids)

(1462, 1462)

In [6]:
fetched, missing = 0, 0
with OUT_FILE.open("a") as out:
    for op_id in new_ids[:MAX_PER_ROUND]:
        posts = get_thread(BOARD, op_id)
        time.sleep(1)                               # API rule: at most one request per second
        seen.add(op_id)                             # do not retry a deleted thread either
        if posts is None:
            missing += 1
            continue
        record = {
            "board": BOARD,
            "thread_id": op_id,
            "collected_at": datetime.now(timezone.utc).isoformat(),
            "posts": posts,                         # the raw response, untouched
        }
        out.write(json.dumps(record) + "\n")
        fetched += 1
save_seen(seen)
fetched, missing, len(seen)

(3, 0, 3)

In [7]:
# What was written
with OUT_FILE.open() as f:
    records = [json.loads(line) for line in f]
len(records), [(r["thread_id"], len(r["posts"])) for r in records]

(3, [(109444547, 315), (109495988, 321), (109497359, 310)])

## Running it for real

- **Schedule it.** Save the cells above as a script and run it from `cron`.
  This line runs it every 10 minutes:

  ```
  */10 * * * * /path/to/python /path/to/collect_4chan.py >> /path/to/collect.log 2>&1
  ```

  The 2022 U.S. midterm dataset used the same design: the catalog every 5
  minutes and the archive every 10.
- **Pick the interval from the data.** Compare two archive polls a few hours
  apart. If IDs disappear from the front of the list faster than you poll,
  you are losing threads.
- **Keep `seen` on disk**, as above. A restart must not refetch everything,
  and it must not lose the IDs it already has.
- **Store the raw response.** Derive fields later. Record when you collected
  each thread, not only when it was posted.
- **Remove `MAX_PER_ROUND`.** The cap is only there so the demo finishes in
  seconds. The first real round fetches the whole archive, about 1,500 threads
  on `/g/`, which takes about 25 minutes at one request per second.
- **Log every round.** A collector that dies overnight and nobody notices is
  the most common failure in Project 1.
- **Images are separate.** The posts carry file names, not files. Download
  images only if your research question needs them, and check the topic page's
  research considerations first.